## h3 core functions

- GeoToH3(lat, lng, res) - takes a point and returns what cell it's in
- H3ToGeo(h3Index) - takes a cell and returnes the geo center (opposite of GeoToH3)
- H3ToGeoBoundary(H3Index, asGeoJson) - takes a cell and returns the corners as geojson

- polyfill(coordinates, res) - convert a polygon to a set of H3 cells
- H3SetToMultiPolygon(H3Indexes) - convert a set of H3 cells to a polygon (opposite of polyfill)

- Clustering?? (when hexes are "grouped" together to draw shapes made of hexes)

## step 1 - generate a map that shows resolution 3 hexagons on a map of the CONUS

## h3 map

Uses **conus-states.json** as datasouce
- contains geometry outline of 48 CONUS states
- no AL, HI or PR
- states are unified into 1 polygon using shapely **unary_union**

Uses h3 **polygon_to_cells_expermental** with **contain="overlap"** (if any part of a hexagon is within poly it is plotted)
- resolution 4 results in 4533 hexagons
- resolution 3 results in 701 hexagons


In [ ]:
import h3
from h3 import LatLngPoly, LatLngMultiPoly
import folium
import json
from shapely.geometry import shape
from shapely.ops import unary_union

resolution = 3

with open('datasets/conus-states.json', 'r') as f:
    # Parsing the JSON file into a Python dictionary
    state_geo = json.load(f)

m = folium.Map(location=(40, -100), zoom_start=5)

geoms = [shape(f["geometry"]) for f in state_geo["features"]]
conus = unary_union(geoms) # conus is shaprely

def convert_shapely_to_h3(geom):
    if geom.geom_type == "Polygon":
        exterior = [(lat, lon) for lon, lat in geom.exterior.coords]
        holes = [
            [(lat, lon) for lon, lat in ring.coords]
            for ring in geom.interiors
        ]
        return LatLngPoly(exterior, holes)

    elif geom.geom_type == "MultiPolygon":
        return LatLngMultiPoly(*(convert_shapely_to_h3(p) for p in geom.geoms))

hexes = h3.polygon_to_cells_experimental(convert_shapely_to_h3(conus), res=resolution, contain="overlap")

for cell in hexes:
    outline = h3.cell_to_boundary(cell)
    folium.Polygon(locations=outline, weight=1, fill=False, color="black").add_to(m)

def add_geom(g):
    if g.geom_type == "Polygon":
        folium.Polygon(
            locations=[(lat, lon) for lon, lat in g.exterior.coords],
            weight=2,
            #color="yellow",
            fill=False
        ).add_to(m)
    elif g.geom_type == "MultiPolygon":
        for p in g.geoms:
            add_geom(p)



add_geom(conus)
m


## Step 2 - find road snapped point closest to center of each hex

Step 3 - find transit distances from each hex to each surrounding